### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.
### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.


In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.3', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2025-08-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x7eeb32526510>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7eeb32526ba0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description="Title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

In [3]:
model_with_structure=model.with_structured_output(Movie)

In [4]:
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.8.3', 'langchain': '1.5.2', 'langchain-groq': '1.2.1'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2025-08-05', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000021B4A79FE00>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021B4A988C20>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title'

In [6]:
model.invoke("provide details about the movie Inception")

AIMessage(content='**Inception**  \n*(also known as *Inception: The Dream‑Heist* in some markets)*\n\n| Category | Detail |\n|----------|--------|\n| **Director** | Christopher\u202fNolan |\n| **Screenwriter** | Christopher\u202fNolan |\n| **Producer(s)** | Christopher\u202fNolan, Emma\u202fThomas, Emma\u202fThomas, Emma\u202fThomas (executive) |\n| **Composer** | Hans\u202fZimmer (score), with additional contributions by Lorne\u202fMiller & The\u202fLark |\n| **Cinematography** | Wally\u202fPfister |\n| **Editor** | Lee\u202fSmith |\n| **Production company** | Syncopy |\n| **Distributor** | Warner\u202fBros. Pictures |\n| **Runtime** | 2\u202fh\u202f28\u202fmin (148\u202fmin) |\n| **Release dates** | 16\u202fJuly\u202f2010 (USA); 9\u202fJuly\u202f2010 (UK) |\n| **Country** | United States / United Kingdom |\n| **Language** | English |\n| **Budget** | $160\u202fmillion (est.) |\n| **Box office** | $836\u202fmillion worldwide |\n| **MPAA rating** | PG‑13 (U.S.) |\n| **Notable accolades*

In [5]:
model_with_structure.invoke("provide details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed structure

In [7]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description="Title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie rating out of 10")

model_with_structure=model.with_structured_output(Movie, include_raw=True)
response=model_with_structure.invoke("provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "The user wants details about the movie Inception. We can use the provided function to get details. The function expects director, rating, title, year. We need to provide those. We'll call functions.Movie with director: Christopher Nolan, rating: maybe 8.8? Actually rating out of 10; we can guess rating 8.8. Title: Inception. Year: 2010. Then return the result.", 'tool_calls': [{'id': 'fc_51469619-83ab-4f70-b6a8-a974f6844f14', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 127, 'prompt_tokens': 157, 'total_tokens': 284, 'completion_time': 0.13852757, 'completion_tokens_details': {'reasoning_tokens': 88}, 'prompt_time': 0.00904899, 'prompt_tokens_details': None, 'queue_time': 0.307688304, 'total_time': 0.14757656}, 'model_name': 'openai/gpt-oss-20b', 'system_finge

## Nested Structure

In [8]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")


In [9]:
model_with_structure=model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160000000.0)

### TypeDict
TyepeDIct provides a simpler alternative using Pythons built in typing ideal when dont need runtime validation.

In [2]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """An example of a TypedDict for movie details."""
    title: Annotated[str, ..., "Title of the movie"]
    year: Annotated[int, ..., "Year of release"]
    director: Annotated[str, ..., "Director of the movie"]
    rating: Annotated[float, ..., "IMDb rating of the movie"]

model_withtypedictStructure=model.with_structured_output(MovieDict)
response=model_withtypedictStructure.invoke("provide details about the movie avengers: endgame")
response

{'director': 'Anthony Russo, Joe Russo',
 'rating': 8.4,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [8]:
from typing import Annotated, TypedDict

class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: Annotated[float | None, "Budget in millions USD"]

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("provide details about the movie Inception")
response


{'budget': 160,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Elliot Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Eames'},
  {'name': 'Ken Watanabe', 'role': 'Saito'},
  {'name': 'Cillian Murphy', 'role': 'Robert Fischer'}],
 'genres': ['Action', 'Sci-Fi', 'Thriller'],
 'title': 'Inception',
 'year': 2010}

In [9]:
model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("provide details about the movie avengers: endgame")
response

{'budget': 356,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Don Cheadle', 'role': 'James Rhodes / War Machine'},
  {'name': 'Paul Rudd', 'role': 'Scott Lang / Ant-Man'},
  {'name': 'Brie Larson', 'role': 'Carol Danvers / Captain Marvel'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'Avengers: Endgame',
 'year': 2019}

In [10]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2025-08-05',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### DataCLasses
A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using @dataclass decorator

In [11]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

In [14]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')


name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [16]:
from typing import Annotated, TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: Annotated[str, "The name of the person"]
    email: Annotated[str, "The email address of the person"]
    phone: Annotated[str, "The phone number of the person"]

agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


In [19]:
from dataclasses import dataclass
from typing import Annotated
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: Annotated[str, "The name of the person"]
    email: Annotated[str, "The email address of the person"]
    phone: Annotated[str, "The phone number of the person"]

agent = create_agent(
    model="google_genai:gemini-3.5-flash",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
